# 00 - Setup e Configuração do Ambiente
**Squad 2 — Real Time for Business | Dupla 5 (Grupo 5)**  
**Integrantes:** Zaiden Emiliano Segundo Seleme *(assumindo o escopo do Grupo 5)*  
**Tabelas de Escopo:** `ecommerce_rastreamento_entregas` e `ecommerce_enderecos`  
**Branch:** `feat/squad2-dupla5-rastreamento-enderecos`  

### Objetivo:
1. Instalação e isolamento das dependências necessárias no Databricks Serverless / Free Edition.
2. Carregamento seguro e resiliente das credenciais via `python-dotenv` com busca multi-nível.
3. Configuração das credenciais OAuth FQDN para o driver Hadoop ABFS (`fs.azure.account.*`).
4. Validação de conectividade com o Azure Data Lake Storage Gen2 (contêineres `raw` e `squad2`).
5. Definição centralizada de caminhos no Data Lake com isolamento de namespace no prefixo `/grupo5/`.

In [ ]:
# 1. Instalação das bibliotecas no escopo da sessão Serverless
%pip install python-dotenv==1.0.1 azure-storage-file-datalake azure-identity pandas pyarrow pyodbc --quiet


In [ ]:
# 2. Reinicialização segura do interpretador Python para carregar novos pacotes
dbutils.library.restartPython()


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# 3. Localização dinâmica e carga do arquivo .env com busca resiliente
dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env"),
        "/Workspace/Shared/estagio_ed",
        "/Workspace/Repos/estagio_ed"
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"Arquivo .env carregado com sucesso de: {dotenv_path}")
else:
    load_dotenv(override=True)
    print("Tentativa de carregar .env a partir das variáveis de ambiente padrão.")

# Leitura das variáveis do ADLS Gen2
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

assert client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"

print("\n=== Status das Credenciais ADLS Gen2 ===")
print(f"  Storage Account: {storage_account}")
print(f"  Client ID:       {'[CONFIGURADO]' if client_id else '[PENDENTE]'}")
print(f"  Tenant ID:       {'[CONFIGURADO]' if tenant_id else '[PENDENTE]'}")
print(f"  Client Secret:   {'[CONFIGURADO]' if client_secret else '[PENDENTE]'}")


In [ ]:
# 4. Configuração OAuth com FQDN do Storage Account para injeção granular no Spark
# Padrão oficial recomendado pela Microsoft/Databricks para contornar restrições do Serverless
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# 5. Definição da malha de caminhos ABFSS (Isolamento de Namespace /grupo5/)
base_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data"
base_squad = f"abfss://squad2@{storage_account}.dfs.core.windows.net/grupo5"

caminhos = {
    "origem_rastreamento": f"{base_raw}/*/*/*/*/ecommerce_rastreamento.parquet",
    "origem_enderecos": f"{base_raw}/*/*/*/*/ecommerce_enderecos.parquet",
    "bronze_rastreamento": f"{base_squad}/bronze/ecommerce_rastreamento_entregas",
    "bronze_enderecos": f"{base_squad}/bronze/ecommerce_enderecos",
    "silver_rastreamento": f"{base_squad}/silver/ecommerce_rastreamento_entregas",
    "silver_enderecos": f"{base_squad}/silver/ecommerce_enderecos",
    "quarantine_rastreamento": f"{base_squad}/quarantine/ecommerce_rastreamento_entregas",
    "quarantine_enderecos": f"{base_squad}/quarantine/ecommerce_enderecos",
    "metadata_control_log": f"{base_squad}/metadata/ingestion_control_log"
}

print("Caminhos configurados para o Grupo 5 (Dupla 5):")
for k, v in caminhos.items():
    print(f"  {k:24}: {v}")


In [ ]:
# 6. Teste de Validação de Conectividade Eager via Leitor Resiliente Serverless
import io
import pyarrow.parquet as pq
import pyarrow as pa
from azure.storage.filedatalake import DataLakeServiceClient
from azure.identity import ClientSecretCredential

schema_rastreamento_pa = pa.schema([
    ('id_rastreamento', pa.int64()),
    ('id_pedido_ecommerce', pa.int64()),
    ('codigo_rastreio', pa.string()),
    ('id_transportadora', pa.int64()),
    ('nome_transportadora', pa.string()),
    ('status_entrega', pa.string()),
    ('dt_evento', pa.timestamp('us')),
    ('observacao', pa.string()),
])

schema_enderecos_pa = pa.schema([
    ('id_endereco', pa.int64()),
    ('id_cliente', pa.int64()),
    ('logradouro', pa.string()),
    ('numero', pa.int64()),
    ('complemento', pa.string()),
    ('bairro', pa.string()),
    ('cep', pa.string()),
    ('cidade', pa.string()),
    ('estado', pa.string()),
    ('latitude', pa.float64()),
    ('longitude', pa.float64()),
    ('apelido', pa.string()),
    ('is_principal', pa.bool_()),
])

def ler_parquet_raw_resiliente(nome_arquivo_alvo, schema_pa):
    """
    Lê os micro-lotes do container raw contornando limitações do Parquet Vectorized Reader
    no Databricks Serverless (trata INT32 Null em campos string e TIMESTAMP_NANOS).
    """
    cred = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)
    service = DataLakeServiceClient(account_url=f"https://{storage_account}.dfs.core.windows.net", credential=cred)
    fs = service.get_file_system_client("raw")
    
    paths = fs.get_paths(path="real-time-data")
    arquivos = [p.name for p in paths if p.name.endswith(nome_arquivo_alvo)]
    
    tables = []
    for p in arquivos:
        fc = fs.get_file_client(p)
        data = fc.download_file().readall()
        t = pq.read_table(io.BytesIO(data))
        if t.num_rows == 0 or len(t.schema.names) == 0:
            continue
        if set(schema_pa.names).issubset(set(t.schema.names)):
            t = t.select(schema_pa.names)
        t_casted = t.cast(schema_pa)
        full_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/{p}"
        t_with_meta = t_casted.append_column("bronze_source_file", pa.array([full_path] * len(t_casted)))
        tables.append(t_with_meta)
        
    if not tables:
        return None
    combined = pa.concat_tables(tables)
    return spark.createDataFrame(combined.to_pandas())


print("\n=== Testando Conectividade com o ADLS Gen2 via Leitor Resiliente ===")

# Teste 1: Leitura na landing zone (raw)
try:
    df_teste_rast = ler_parquet_raw_resiliente("ecommerce_rastreamento.parquet", schema_rastreamento_pa)
    df_teste_end = ler_parquet_raw_resiliente("ecommerce_enderecos.parquet", schema_enderecos_pa)
    print(f"[SUCESSO] Conectado ao container 'raw'. Rastreamento ({df_teste_rast.count()} linhas) e Endereços ({df_teste_end.count()} linhas).")
except Exception as e_raw:
    print(f"[FALHA] Erro ao ler container 'raw': {e_raw}")

# Teste 2: Verificação do namespace squad2/grupo5
def tabela_delta_existe(caminho):
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False

status_bronze_rastreio = tabela_delta_existe(caminhos["bronze_rastreamento"])
status_bronze_enderecos = tabela_delta_existe(caminhos["bronze_enderecos"])

print(f"Status inicial do Bronze Rastreamento: {'[EXISTENTE]' if status_bronze_rastreio else '[INICIAL - SERA CRIADO]'}")
print(f"Status inicial do Bronze Endereços:    {'[EXISTENTE]' if status_bronze_enderecos else '[INICIAL - SERA CRIADO]'}")
print("\nSetup da Dupla 5 concluído com sucesso e pronto para os pipelines de dados!")
